# Test python verification packages with real dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
print("Python version")
print (sys.version)
#print("Version info.")
#print (sys.version_info)

Python version
3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44) [GCC 11.3.0]


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import scoringrules
from scores.probability import crps_for_ensemble
import properscoring
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_verification/')
import vrf_scores
import pymvscore

In [4]:
## Load dataset
streamflow = xr.open_dataset('stremflow_data_for_scores.nc')
streamflow

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB ...
    fcst        (time, ens_member, station) float32 12MB ...

## Run scores

In [41]:
from numba import guvectorize, float32

In [ ]:
@guvectorize([(float32[:], int64, int64[:])], '(n),()->(n)')
def g(x, y, res):
    for i in range(x.shape[0]):
        res[i] = x[i] + y

In [45]:
#from numba import jit
@guvectorize([(float32[:,:], float32[:], float32[:])], '(n),()->(n)')
#@guvectorize([(float32[:], int64, int64[:])], '(n),()->(n)')
def gnu_crps(pred, obs,res):
    #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    dim=0
    n = pred.shape[dim]
    pred = np.sort(pred, axis=dim)
    ans = np.zeros_like(obs)

    # dx [F(x) - H(x-y)]^2 = dx [0 - 1]^2 = dx
    # val = ensemble[0] - truth
    val = (pred[0, :] - obs)
    #val = (pred[:, 0] - obs)
    ans += np.maximum(val, 0.0)

    for i in range(n - 1):
        x0 = pred[i, :]
        x1 = pred[i+1, :]

        cdf = (i + 1) / n

        # a. case y < x0
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = obs < x0
        ans += val * mask

        # b. case x0 <= y <= x1
        val = (obs - x0) * cdf**2 + (x1 - obs) * (cdf - 1) ** 2
        mask = (obs >= x0) & (obs <= x1)
        ans += val * mask

        # c. case x1 < t
        mask = obs > x1
        val = (x1 - x0) * cdf**2
        ans += val * mask

    # dx [F(x) - H(x-y)]^2 = dx [1 - 0]^2 = dx
    val = obs - pred[-1, :]
    ans += np.maximum(val, 0.0)
    res[0] = ans
    

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mNo implementation of function Function(<function sort at 0x1555501cb7f0>) found for signature:
 
 >>> sort(array(float32, 2d, A), axis=Literal[int](0))
 
There are 2 candidate implementations:
[1m  - Of which 2 did not match due to:
  Overload in function 'impl_np_sort': File: numba/np/arrayobj.py: Line 6423.
    With argument(s): '(array(float32, 2d, A), axis=int64)':[0m
[1m   Rejected as the implementation raised a specific error:
     TypingError: [1mgot an unexpected keyword argument 'axis'[0m[0m
  raised from /home/shr015/.local/lib/python3.10/site-packages/numba/core/typing/templates.py:783
[0m
[0m[1mDuring: resolving callee type: Function(<function sort at 0x1555501cb7f0>)[0m
[0m[1mDuring: typing of call at /tmp/ipykernel_107413/2246872519.py (8)
[0m
[1m
File "../../../../../../../../tmp/ipykernel_107413/2246872519.py", line 8:[0m
[1m<source missing, REPL/exec in use?>[0m


In [5]:
#from numba import jit
#@jit(nopython=True)
def crps_from_empirical_cdf(pred, obs, dim=0):
    #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    n = pred.shape[dim]
    pred = np.sort(pred, axis=dim)
    ans = np.zeros_like(obs)

    # dx [F(x) - H(x-y)]^2 = dx [0 - 1]^2 = dx
    # val = ensemble[0] - truth
    val = (pred[0, :] - obs)
    #val = (pred[:, 0] - obs)
    ans += np.maximum(val, 0.0)

    for i in range(n - 1):
        x0 = pred[i, :]
        x1 = pred[i+1, :]

        cdf = (i + 1) / n

        # a. case y < x0
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = obs < x0
        ans += val * mask

        # b. case x0 <= y <= x1
        val = (obs - x0) * cdf**2 + (x1 - obs) * (cdf - 1) ** 2
        mask = (obs >= x0) & (obs <= x1)
        ans += val * mask

        # c. case x1 < t
        mask = obs > x1
        val = (x1 - x0) * cdf**2
        ans += val * mask

    # dx [F(x) - H(x-y)]^2 = dx [1 - 0]^2 = dx
    val = obs - pred[-1, :]
    ans += np.maximum(val, 0.0)
    return ans

In [6]:
def energy_score(forecasts, obs):

    # must have dimensions of (num_variables, num_ensembles). Variables can include different locations, time steps, climate variables etc

    num_samples = forecasts.shape[1]

    s1 = np.sqrt(np.sum(np.square(forecasts - obs[:, np.newaxis]), axis=0)).sum()

    pairwise_diffs = forecasts[:, :, np.newaxis] - forecasts[:, np.newaxis, :]
    s2 = np.sqrt(np.sum(np.square(pairwise_diffs), axis=0)).sum()

    es = (s1 / num_samples) - s2 / (2 * num_samples**2)
    
    return es

In [7]:
def energy_score_multiple(f_ens, o,return_mean=True):
    
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            crps.append(energy_score(ff, oo))   # obs should be 
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return energy_score(f_ens, o)  

In [8]:
def mvscore_multiple(f_ens, o,return_mean=True):
    ms = pymvscore.PyMultivariateScore()
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            #crps.append(energy_score(ff, oo))   # obs should be 
            crps.append(ms.crpsECDF_many(ff, oo))
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return ms.crpsECDF_many(f_ens, o)  

In [9]:
def remove_nans(fcst,obs,site):
    ff = fcst.isel(station=site).values
    oo = obs.isel(station=site).values   
    mean_fcst = np.mean(ff,axis=1) 
    nanindx = ~np.isnan(oo) & ~np.isnan(mean_fcst)
    oo = oo[nanindx]
    ff = ff[nanindx,:] 
    return ff,oo

In [10]:
nsite = streamflow['fcst'].shape[2]
nsite

16

In [11]:
streamflow

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB ...
    fcst        (time, ens_member, station) float32 12MB ...

### 1. scores.probability.crps_for_ensemble

In [11]:
%%time
# Specifying method='ecdf' assumes the empirical CDF
scores_crps = crps_for_ensemble(streamflow['fcst'], streamflow['obs'], ensemble_member_dim='ens_member', method='ecdf',preserve_dims='station')

CPU times: user 540 ms, sys: 268 ms, total: 807 ms
Wall time: 969 ms


### 2. scoringrules.crps_ensemble

In [12]:
%%time
ens_mem_dim = streamflow['fcst'].get_axis_num("ens_member")
scoringrules_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    scoringrules_crps_vals = scoringrules.crps_ensemble(obs,fcst, axis=1, estimator="nrg")
    scoringrules_crps.append(scoringrules_crps_vals.mean().item())

CPU times: user 156 ms, sys: 88 ms, total: 244 ms
Wall time: 243 ms


### 3. properscoring.crps_ensemble

In [13]:
%%time
properscoring_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    properscoring_crps.append(np.mean(properscoring.crps_ensemble(obs,fcst)) )    

CPU times: user 64.7 ms, sys: 42 µs, total: 64.7 ms
Wall time: 95.4 ms


### 4. vrf_scores.crps_ecdf_multiple

In [14]:
%%time
vrf_scores_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    vrf_scores_crps.append(vrf_scores.crps_ecdf_multiple(fcst,obs)) 

CPU times: user 4.63 s, sys: 16.2 ms, total: 4.65 s
Wall time: 4.65 s


### 5. crps_from_empirical_cdf

In [15]:
%%time
ecdf_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    ecdf_crps.append(np.mean(crps_from_empirical_cdf(fcst.T,obs,dim=0))) 

CPU times: user 88.3 ms, sys: 4.02 ms, total: 92.4 ms
Wall time: 91.5 ms


### 6. Energy score

In [16]:
%%time
eng_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    eng_crps.append(energy_score_multiple(fcst,obs)) 

CPU times: user 1.22 s, sys: 0 ns, total: 1.22 s
Wall time: 1.22 s


### 7. MVscore

In [37]:
%%time
mvscore_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    mvscore_crps.append(mvscore_multiple(fcst,obs)) 

CPU times: user 336 ms, sys: 0 ns, total: 336 ms
Wall time: 336 ms


In [38]:
%%time
mvscore_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)   
    mvscore_crps.append(ms.crpsECDF_many(fcst, obs))

CPU times: user 226 ms, sys: 0 ns, total: 226 ms
Wall time: 226 ms


In [39]:
%%time
mvscore_crps = []
for s in range(1):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    mvscore_crps.append(mvscore_multiple(fcst,obs)) 

CPU times: user 15.2 ms, sys: 10.8 ms, total: 26 ms
Wall time: 25.1 ms


In [40]:
%%time
mvscore_crps = []
for s in range(1):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)   
    mvscore_crps.append(ms.crpsECDF_many(fcst, obs))

CPU times: user 16.8 ms, sys: 44 µs, total: 16.9 ms
Wall time: 16.7 ms


In [36]:
mvscore_crps

[0.6658845880100637,
 5.458856096315852,
 1.1149853531849891,
 0.5313843076093623,
 3.1841417898583733,
 2.375741629164671,
 2.8585770512631004,
 13.33198621722825,
 36.13853519210953,
 35.97743880576046,
 47.71221368041656,
 38.966512091808056,
 23.099913891183103,
 415.9833147162473,
 40.126628691573664,
 31.135090645592076]

### make dataframe

In [20]:
df = pd.DataFrame(scores_crps.values,columns = ['scores.crps(871 ms)'])
df['scoringrules(243 ms)'] = scoringrules_crps
df['properscoring(95.4 ms)'] = properscoring_crps
df['vrf_scores(4.65 s)']  = vrf_scores_crps
df['ecdf(91.5 ms)'] = ecdf_crps
df['eng(1.22 s)'] = eng_crps
df['mvscore(331 ms)'] = mvscore_crps
df.index.name = 'sites'
df

,scores.crps(871 ms),scoringrules(243 ms),properscoring(95.4 ms),vrf_scores(4.65 s),ecdf(91.5 ms),eng(1.22 s),mvscore(331 ms)
sites,,,,,,,
0,0.665885,0.665885,0.665885,0.665885,0.665885,0.665885,0.665885
1,5.458856,5.458856,5.458856,5.458856,5.458857,5.458856,5.458856
2,1.114985,1.114985,1.114985,1.114985,1.114985,1.114985,1.114985
3,0.531385,0.531384,0.531384,0.531384,0.531384,0.531384,0.531384
4,3.184142,3.184142,3.184142,3.184142,3.184142,3.184142,3.184142
5,2.375742,2.375742,2.375742,2.375742,2.375742,2.375742,2.375742
6,2.858582,2.858577,2.858577,2.858577,2.858577,2.858577,2.858577
7,13.331986,13.331985,13.331986,13.331986,13.331985,13.331986,13.331986
8,36.138537,36.138538,36.138535,36.138535,36.138535,36.138535,36.138535


## Energy_Score

In [11]:
num_ens = 500 
ens1, obs1 = np.random.normal(200,20,num_ens), 150.
ens2, obs2 = np.random.normal(150,30,num_ens), 170.
forecasts = [ens1, ens2]
obs = [obs1, obs2]

In [9]:
%%timeit
#crps_scores = [crps_from_empirical_cdf(e, o) for e,o in zip(forecasts,obs)]
energy_scores = [energy_score(np.array([e]), np.array([o])) for e,o in zip(forecasts,obs)]
for e,o in zip(forecasts,obs):
    energy_scores1 = energy_score(np.array([e]), np.array([o]))
#print('crps_ecdf:       ', crps_scores, '  mean:', np.mean(crps_scores))
#print('optimised energy:', energy_scores, '  mean:', np.mean(energy_scores))

6.29 ms ± 32.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## MVScore

In [10]:
%%timeit
#num_ens = 500
#ens1, obs1 = np.random.normal(200,20,num_ens), 150.
#ens2, obs2 = np.random.normal(150,30,num_ens), 170.
#forecasts = [ens1, ens2]
#obs = [obs1, obs2] 
ms = pymvscore.PyMultivariateScore()
score_cpp = ms.crpsECDF_many(forecasts, obs)

46.5 µs ± 391 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [5]:
score_cpp

24.604201519426898

In [12]:
forecasts

[array([192.8057262 , 211.35051068, 209.8577137 , 228.71174287,
        199.98205134, 191.523498  , 226.62830555, 164.1282447 ,
        166.27768826, 178.52102982, 238.55930851, 223.86601436,
        157.47928085, 209.70682509, 219.07156616, 183.99653158,
        203.96044318, 171.84390297, 185.31328432, 226.66109381,
        182.41599123, 207.55757026, 191.83515413, 191.76028357,
        196.62440904, 172.27372899, 176.10709554, 198.04189423,
        187.73707988, 171.10017782, 191.71797269, 172.87194775,
        195.00067756, 186.3923851 , 229.24375754, 182.07065055,
        185.52673302, 195.59927767, 173.18802501, 191.22910642,
        212.83243645, 221.99706196, 231.44188666, 185.27754072,
        210.09092174, 243.32626854, 215.19649168, 198.8118459 ,
        211.93087061, 203.41250206, 229.23618504, 196.00831262,
        189.67045459, 182.84244095, 185.3262983 , 210.75036841,
        213.24431918, 241.36509551, 227.11608469, 211.15960109,
        220.56805354, 179.67110813, 182.

In [20]:
def mvscore_multiple(f_ens, o,return_mean=True):
    ms = pymvscore.PyMultivariateScore()
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            #crps.append(energy_score(ff, oo))   # obs should be 
            crps.append(ms.crpsECDF_many(ff, oo))
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return ms.crpsECDF_many(f_ens, o)  

In [21]:
%%time
mvscore_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    mvscore_crps .append(mvscore_multiple(fcst,obs)) 

CPU times: user 349 ms, sys: 12.2 ms, total: 361 ms
Wall time: 360 ms


In [22]:
mvscore_crps

[0.6658845880100632,
 5.458856096315842,
 1.1149853531849896,
 0.5313843076093614,
 3.1841417898583764,
 2.375741629164669,
 2.858577051263099,
 13.331986217228263,
 36.13853519210955,
 35.97743880576042,
 47.7122136804166,
 38.96651209180809,
 23.099913891183093,
 415.98331471624806,
 40.126628691573686,
 31.135090645592086]

In [23]:
fcst = streamflow['fcst'].isel(station=0).values
fcst.shape

(3743, 50)

In [24]:
obs = streamflow['obs'].isel(station=0).values
obs.shape

(3743,)

In [25]:
# Find rows where obs or any element in forecasts has NaN
nonnan_rows = ~np.isnan(obs) & ~np.isnan(fcst).any(axis=1)
# Filter out rows with NaNs
fcst = fcst[nonnan_rows, :]
obs = obs[nonnan_rows]

In [26]:
fcst.shape,obs.shape

((3740, 50), (3740,))

In [28]:

ms = pymvscore.PyMultivariateScore()
mvscore_crps = ms.crpsECDF_many(fcst, obs)

In [29]:
mvscore_crps

0.6658845880100637

# Scaling check

### Testing on same dataset

In [80]:
np.random.seed(42)  # Set the seed to an integer value
num_events = 3742
num_ens = [50,5000,50000];
obs =  np.random.normal(150,1,num_events) 

### Ensemble size of 50

In [81]:
fcst = np.random.normal(150,30,(num_events,num_ens[0])) 
fcstT = fcst.T

In [82]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

6.56 ms ± 10.1 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [83]:
%%timeit
ms.crpsECDF_many(fcst, obs);

14.5 ms ± 54.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Ensemble size of 5000

In [84]:
fcst = np.random.normal(150,30,(num_events,num_ens[1]))
fcstT = fcst.T

In [85]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

1.22 s ± 4.24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [86]:
%%timeit
ms.crpsECDF_many(fcst, obs);

1.66 s ± 40.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Ensemble size of 50000

In [87]:
fcst = np.random.normal(150,30,(num_events,num_ens[2]))
fcstT = fcst.T

In [88]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

15 s ± 21.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [89]:
%%timeit
ms.crpsECDF_many(fcst, obs);

18.6 s ± 86.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [90]:
import psutil
# Number of physical cores
physical_cores = psutil.cpu_count(logical=False)
# Number of logical processors (threads)
logical_processors = psutil.cpu_count(logical=True)

In [91]:
physical_cores

64

In [92]:
logical_processors

128

In [93]:
import multiprocessing
# Number of available CPU cores
num_cores = multiprocessing.cpu_count()

In [94]:
num_cores

128

### Testing on size of num events

In [16]:
ms = pymvscore.PyMultivariateScore()

In [12]:
np.random.seed(42)  # Set the seed to an integer value
num_events = [374,3742,37420]
num_ens = 5000;

### Num events of 374, ens size 5000

In [24]:
obs =  np.random.normal(150,1,num_events[0]) 
fcst = np.random.normal(150,30,(num_events[0],num_ens)) 
fcstT = fcst.T

In [27]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

175 ms ± 452 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
%%timeit
ms.crpsECDF_many(fcst, obs);

160 ms ± 267 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Num events of 3740, ens size 5000

In [29]:
fcst = np.random.normal(150,30,(num_events[1],num_ens))
obs =  np.random.normal(150,1,num_events[1]) 
fcstT = fcst.T

In [32]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

1.24 s ± 21.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
%%timeit
ms.crpsECDF_many(fcst, obs);

1.65 s ± 9.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### ### Num events of 37400, ens size 5000

In [34]:
fcst = np.random.normal(150,30,(num_events[2],num_ens))
obs =  np.random.normal(150,1,num_events[2]) 
fcstT = fcst.T

In [37]:
%%timeit
np.mean(crps_from_empirical_cdf(fcstT,obs,dim=0));

11.9 s ± 113 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [38]:
%%timeit
ms.crpsECDF_many(fcst, obs);

16.5 s ± 68.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
obs.shape

(37420,)

In [36]:
fcst.shape

(37420, 5000)

In [40]:
ds = xr.open_dataset('197912312300.CHRTOUT_DOMAIN1')
ds

<xarray.Dataset> Size: 200MB
Dimensions:         (feature_id: 2776734, time: 1, reference_time: 1)
Coordinates:
  * time            (time) datetime64[ns] 8B 1979-12-31T23:00:00
  * reference_time  (reference_time) datetime64[ns] 8B 1979-10-01
  * feature_id      (feature_id) int64 22MB 101 179 ... 1180001803 1180001804
    latitude        (feature_id) float32 11MB ...
    longitude       (feature_id) float32 11MB ...
Data variables:
    qBtmVertRunoff  (feature_id) float64 22MB ...
    qBucket         (feature_id) float64 22MB ...
    qSfcLatRunoff   (feature_id) float64 22MB ...
    q_lateral       (feature_id) float64 22MB ...
    streamflow      (feature_id) float64 22MB ...
    velocity        (feature_id) float64 22MB ...
    crs             |S1 1B ...
    order           (feature_id) int32 11MB ...
    elevation       (feature_id) float32 11MB ...
Attributes: (12/20)
    TITLE:                      OUTPUT FROM WRF-Hydro v5.3.0-alpha1
    featureType:                timeSeries
    proj4:                      +proj=lcc +units=m +a=6370000.0 +b=6370000.0 ...
    model_initialization_time:  1979-10-01_00:00:00
    station_dimension:          feature_id
    model_output_valid_time:    1979-12-31_23:00:00
    ...                         ...
    dev_NOAH_TIMESTEP:          3600
    dev_channel_only:           0
    dev_channelBucket_only:     0
    dev:                        dev_ prefix indicates development/internal me...
    NCO:                        netCDF Operators version 5.1.4 (Homepage = ht...
    history:                    Fri Sep 15 17:43:58 2023: ncatted -O -a missi...